In [1]:
import pandas as pd
import os

reviews = pd.read_csv('../data/raw/IMDB Dataset.csv')
print(f"Reviews found: {len(reviews)} rows")
print(f"Columns: {reviews.columns.tolist()}")

Reviews found: 50000 rows
Columns: ['review', 'sentiment']


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

True

In [3]:
print("=" * 60)
print("SENTIMENT ANALYSIS MODEL TRAINING")
print("=" * 60)

# Check sentiment distribution
print("\nSentiment distribution:")
print(reviews['sentiment'].value_counts())
print(f"\nSample review:")
print(f"  Sentiment: {reviews['sentiment'].iloc[0]}")
print(f"  Text: {reviews['review'].iloc[0][:200]}...")

SENTIMENT ANALYSIS MODEL TRAINING

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample review:
  Sentiment: positive
  Text: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...


In [4]:
# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    reviews['review'],
    reviews['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=reviews['sentiment']
)

print(f"\nTraining set: {len(X_train):,} reviews")
print(f"Test set: {len(X_test):,} reviews")


Training set: 40,000 reviews
Test set: 10,000 reviews


In [5]:
# TF-IDF Vectorization

print("\n" + "-" * 40)
print("Training sentiment TF-IDF vectorizer...")

sentiment_tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.8,
    stop_words='english'
)

X_train_vec = sentiment_tfidf.fit_transform(X_train)
X_test_vec = sentiment_tfidf.transform(X_test)


print(f"Vocabulary size: {len(sentiment_tfidf.vocabulary_)}")
print(f"Training matrix shape: {X_train_vec.shape}")
print(f"Test matrix shape: {X_test_vec.shape}")


----------------------------------------
Training sentiment TF-IDF vectorizer...
Vocabulary size: 10000
Training matrix shape: (40000, 10000)
Test matrix shape: (10000, 10000)


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("\n" + "-" * 40)
print("Training sentiment classifier...")

sentiment_model = LogisticRegression(
    C=1.0,
    random_state=42,
    max_iter=1000,
    # class_weight='balanced'
)

sentiment_model.fit(X_train_vec, y_train)

# Evaluate

y_pred = sentiment_model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest accuracy: {accuracy:.4f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))


----------------------------------------
Training sentiment classifier...

Test accuracy: 0.8969%

Classification Report:
              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      5000
    positive       0.89      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [9]:
# Save Models

print("\n" + "-" * 40)
print("Saving sentiment models...")

with open('../models/sentiment/sentiment_vectorizer.pkl', 'wb') as f:
    pickle.dump(sentiment_tfidf, f)
print("Saved: Sentiment Vectorizer")


# Save classifier
with open('../models/sentiment/sentiment_classifier.pkl', 'wb') as f:
    pickle.dump(sentiment_model, f)
print("Saved: Sentiment Classifier")


----------------------------------------
Saving sentiment models...
Saved: Sentiment Vectorizer
Saved: Sentiment Classifier


In [ ]:
# MLflow Tracking
import mlflow
from datetime import datetime

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("Movie_Recommendation_Experiments")

with mlflow.start_run(run_name=f"sentiment_logreg_{datetime.now().strftime('%Y%m%d_%H%M')}"):
    
    # Parameters
    mlflow.log_param("vectorizer", "TfidfVectorizer")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("classifier", "LogisticRegression")
    mlflow.log_param("C", 1.0)
    
    # Metrics
    mlflow.log_metric("accuracy", accuracy)
    
    # Artifacts
    mlflow.log_artifact("../models/sentiment/sentiment_vectorizer.pkl")
    mlflow.log_artifact("../models/sentiment/sentiment_classifier.pkl")
    
    # Tags
    mlflow.set_tag("model_type", "sentiment_analysis")
    mlflow.set_tag("status", "production_ready")

print("\n✓ Sentiment training complete!")


✓ Sentiment training complete!


In [22]:
# Test sentiment model on sample reviews

print("\n" + "=" * 40)
print("TESTING SENTIMENT MODEL")
print("=" * 40)


test_reviews = [
    "This movie was absolutely amazing! Best film ever!",
    "Terrible waste of time. Boring and predictable.",
    "It was okay, nothing special but not terrible.",
    "The acting was superb but the plot was weak."
]

for reviews in test_reviews:
    vec = sentiment_tfidf.transform([reviews])
    pred = sentiment_model.predict(vec)[0]
    proba = sentiment_model.predict_proba(vec).max()
    print(f"\nReveiws: {reviews}")
    print(f"\nPredict Sentiment: {pred} (confidence: {proba:2f})")


TESTING SENTIMENT MODEL

Reveiws: This movie was absolutely amazing! Best film ever!

Predict Sentiment: positive (confidence: 0.960100)

Reveiws: Terrible waste of time. Boring and predictable.

Predict Sentiment: negative (confidence: 0.999980)

Reveiws: It was okay, nothing special but not terrible.

Predict Sentiment: negative (confidence: 0.986116)

Reveiws: The acting was superb but the plot was weak.

Predict Sentiment: negative (confidence: 0.507500)
